<a href="https://colab.research.google.com/github/johanndebodastudent/ai-social-good-project/blob/main/MilestoneTwo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Section 1: Problem and Population

The issue in question that we are going over is regarding homelessness. These homeless individuals face the hardship of high barrier to entry for jobs, employment layoffs, and high living/housing costs that keep these people in poverty and homelessness. Alongside this, by not having specific actions (specifically by AI) to not only remedy any issues caused by homeless individuals but also remedy them from being in poverty, it makes it hard for this cycle of poverty into issues to stop. This connects to the UN's first SDG being "No poverty". Pertaining to M1 the biggest change is more so not necessarily targetting how to solve this issue as a whole, but modifying AI systems to be able to recognize situations related to this and creating action plans accordingly through distinguishing the exact issues and not confusing it with other issues instead of trying to solve a huge  global issue with many differing solutions as it would be more feasible to make steps towards this one at a time.

# Section 2

INPUT : Individuals or homeowners reporting to San Jose's 311 system within an added "homelessness" branch about an issue pertaining to or related to homeless individuals

AI Processing: The AI system derives info from the call/message according to this classification : (location, problem caused, urgency, department to be called, and resident language)

Output: From this, the AI system contacts a human and tells them a brief overview of the situation, the info they have derived, and its suggestions as to what departments to call alongside what possible actions should be taken such as who else to contact.

Real-World Action : From this, the department comes and takes action using their experience, critical-thinking skills, and AI reccomendations in mind to be able to provide by contacting the right individuals which are suggested by AI in order to provide the best remedy to the situation taking extra preventative measures that would help in the long-term rather than focus on just solving the short-term issue.



# Section 3

In [1]:
!pip install -q google-generativeai
import google.generativeai as genai
from google.colab import userdata
import json
import time
from collections import Counter


genai.configure(api_key=userdata.get('GEMINI_API_KEY'))


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [2]:
resident_message = """
I am calling about an issue near the Guadalupe River Trail by W San Carlos Street.
There are several tents close to the walking path, and some trash is starting to build up near the sidewalk.
The path is still partly usable, but it is becoming harder for pedestrians to pass safely.
I am not sure which department should handle this, but I think the city should check on both the cleanup issue
and whether the people staying there may need outreach support.
"""

Import needed extensions + call message prompt

In [3]:
schema_prompt = """
Extract information from this San Jose 311 resident complaint.

Return ONLY valid JSON with exactly these six fields:

{
  "location": string,
  "reported_issue": string,
  "urgency": "LOW" or "MEDIUM" or "HIGH",
  "primary_responder": string,
  "supporting_services": list of strings,
  "resident_language": string
}

Field guide:
- location: the street address, park, intersection, or location described
- reported_issue: the civic, safety, environmental, or service-related issue described
- urgency: LOW, MEDIUM, or HIGH
- primary_responder: the first human department or service that should review/respond first
- supporting_services: other services the primary responder may consider contacting only if supported by the report, such as "Homeless Outreach Services", "Emergency Medical Services", "Environmental Services", "Public Works", "Code Enforcement", or "Parks & Recreation"- resident_language: language the resident wrote in, such as "English", "Spanish", or "Vietnamese"

Urgency guide:
LOW = minor litter, general concern, or non-urgent service request
MEDIUM = ongoing issue, blocked access, large debris, or situation requiring city follow-up
HIGH = hazardous materials, immediate safety risk, medical risk, fire risk, or violence

Do not blame people experiencing homelessness unless the report explicitly provides evidence.
The AI should not directly provide treatment, enforcement, or final decisions. It should identify the primary human responder and possible supporting services for human review.
No explanation. No markdown. JSON only.
"""

This prompt is used to provide the classification for AI when it derives and reads through the call in order to get the exact info that is needed but also to be able to achieve and maintain consistency,

In [4]:
def extract_structured(message):
    m = genai.GenerativeModel(
        model_name="gemini-2.5-flash",
        system_instruction=schema_prompt
    )

    response = m.generate_content(message)
    time.sleep(12)  # stays under free tier rate limit

    raw = response.text.strip()

    # Strip markdown code fences if present
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()

    return json.loads(raw)


print("=== STRUCTURED OUTPUT ===")

try:
    final_result = extract_structured(resident_message)
    print(json.dumps(final_result, indent=2, ensure_ascii=False))

except Exception:
    final_result = {
        "location": "Guadalupe River Trail by W San Carlos Street",
        "reported_issue": "Several tents close to the walking path and trash buildup near the sidewalk, making it harder for pedestrians to pass safely.",
        "urgency": "MEDIUM",
        "primary_responder": "Homeless Outreach Services",
        "supporting_services": ["Public Works", "Parks & Recreation"],
        "resident_language": "English"
    }

    print("Used saved output because the Gemini API quota/rate limit was reached.")
    print(json.dumps(final_result, indent=2, ensure_ascii=False))

=== STRUCTURED OUTPUT ===
{
  "location": "Guadalupe River Trail by W San Carlos Street",
  "reported_issue": "Several tents are close to the walking path, and trash is accumulating near the sidewalk, making it difficult for pedestrians to pass safely.",
  "urgency": "MEDIUM",
  "primary_responder": "Homeless Outreach Services",
  "supporting_services": [
    "Public Works",
    "Parks & Recreation"
  ],
  "resident_language": "English"
}


This runs the extraction 3 times to be able to ensure consistency and eliminate hallucinations as given the emergency aspect of these situations, it will be needed to ensure no hallucinations or mistake occur

In [5]:
def analyze_call_all(message):
    m = genai.GenerativeModel(
        model_name="gemini-2.5-flash"
    )

    prompt = f"""
You are analyzing a San Jose 311 call/message.

Resident call/message:
{message}

Answer these four sections clearly:

1. PROBLEM:
Describe the civic, safety, environmental, or service-related issue. Be specific and avoid blaming people unless there is clear evidence.

2. IMPACT:
Describe the public health, safety, service, or community impacts based only on the call/message.

3. URGENCY:
Rate urgency as LOW, MEDIUM, or HIGH. Explain your rating in 2–3 sentences.

4. ACTION:
Which primary human responder should review this first, and what supporting services might they consider contacting based only on the report? Do not assume services without evidence. What precautions should they take?
"""

    response = m.generate_content(prompt)
    time.sleep(12)

    return response.text.strip()


try:
    civic_analysis = analyze_call_all(resident_message)

    print("--- CIVIC ANALYSIS ---")
    print(civic_analysis)

except Exception:
    print("--- CIVIC ANALYSIS ---")
    print("Skipped because the Gemini API quota/rate limit was reached. The structured extraction and resident response sections still demonstrate the system pipeline.")

--- CIVIC ANALYSIS ---
Here is an analysis of the San Jose 311 call:

---

1.  **PROBLEM:**
    There is an unauthorized encampment consisting of several tents located close to the walking path on the Guadalupe River Trail near W San Carlos Street. Concurrently, trash is accumulating near the sidewalk in the same vicinity. The combination of tents and trash is obstructing the public walking path.

2.  **IMPACT:**
    The accumulation of tents and trash is making it increasingly difficult for pedestrians to pass safely on the walking path. While the path is still partly usable, its overall safety and accessibility are diminishing, affecting the public's ability to utilize this civic infrastructure as intended.

3.  **URGENCY:**
    **MEDIUM.** The issue is described as "starting to build up" and "becoming harder," indicating a developing problem rather than an immediate critical emergency. However, it directly impacts public safety by hindering safe pedestrian passage on a public trail 

This part serves to analyze the situation through four ways concising what it had derived initially in order to prepare itself from creating a response.

In [6]:
multilingual_prompt_active = """
You are a helpful assistant for the San Jose 311 civic reporting system.

A resident has submitted a 311 report. Write a brief, polite acknowledgment that:
1. Confirms the report was received.
2. Names the reported issue.
3. Explains that the report will be reviewed by a human dispatcher or city staff member.
4. States that the report will be reviewed and that appropriate services may be considered if supported by the information provided.
5. Gives an expected response timeframe of 3–5 business days for non-emergency issues.

Important:
- Do not promise that a specific department will respond.
- Do not provide treatment, enforcement, or final decisions.
- Do not blame people experiencing homelessness.
- If the report describes immediate danger, tell the resident to contact emergency services.
- Keep the response under 80 words.
- Detect the resident's language and respond in that same language.
"""

def generate_resident_response(message, structured_result):
    m = genai.GenerativeModel(
        model_name="gemini-2.5-flash",
        system_instruction=multilingual_prompt_active
    )

    prompt = f"""
Resident message:
{message}

Structured report:
{json.dumps(structured_result, indent=2, ensure_ascii=False)}
"""

    response = m.generate_content(prompt)
    time.sleep(12)

    return response.text.strip()


resident_reply = generate_resident_response(resident_message, final_result)

print("--- Resident Acknowledgment ---")
print(resident_reply)

--- Resident Acknowledgment ---
Thank you for your report regarding several tents close to the Guadalupe River Trail walking path and accumulating trash. Your report has been received and will be reviewed by a city staff member. Appropriate services may be considered based on the information provided. For non-emergency issues, please expect a response within 3-5 business days.


This part creates the the framework for how the AI should response to ensure consistency, and it also prints the output, being the response.

# Section 4: Edge Case Elicitation

In [7]:
edge_case_message = """
There are two people sitting near the park entrance with bags next to them.
They might be homeless, but I am not sure. They are not blocking anyone, and I do not see trash or danger.
The area just feels uncomfortable.
"""

print("=== STRUCTURED OUTPUT ===")
try:
    edge_result = extract_structured(edge_case_message)
    print(json.dumps(edge_result, indent=2, ensure_ascii=False))
except Exception:
    print("Skipped due to API quota/rate limit.")
    edge_result = {}
print()

print("=== CIVIC ANALYSIS ===")
try:
    edge_civic_analysis = analyze_call_all(edge_case_message)
    print(edge_civic_analysis)
except Exception:
    print("Skipped due to API quota/rate limit.")
print()

print("=== RESIDENT RESPONSE ===")
try:
    if edge_result:
        edge_reply = generate_resident_response(edge_case_message, edge_result)
        print(edge_reply)
    else:
        print("Skipped due to missing structured output.")
except Exception:
    print("Skipped due to API quota/rate limit.")

=== STRUCTURED OUTPUT ===
{
  "location": "Park entrance",
  "reported_issue": "People sitting near park entrance causing discomfort",
  "urgency": "LOW",
  "primary_responder": "Parks & Recreation",
  "supporting_services": [
    "Homeless Outreach Services"
  ],
  "resident_language": "English"
}

=== CIVIC ANALYSIS ===
Here is an analysis of the San Jose 311 call:

1.  **PROBLEM:**
    The core problem is a resident's discomfort due to the presence of two individuals near a park entrance who they believe might be experiencing homelessness. While the individuals are not causing any immediate danger, blocking access, or creating trash, their presence is making the area "feel uncomfortable" to the caller. This represents a perceived issue of public order and community comfort rather than a direct civic violation or safety threat.

2.  **IMPACT:**
    Based solely on the call/message, the primary impact is a **community comfort/perception impact**. The resident explicitly states, "The a

=== STRUCTURED OUTPUT ===
{
  "location": "Park entrance",
  "reported_issue": "People sitting near park entrance causing discomfort",
  "urgency": "LOW",
  "primary_responder": "Parks & Recreation",
  "supporting_services": [
    "Homeless Outreach Services"
  ],
  "resident_language": "English"
}

=== CIVIC ANALYSIS ===
Here is an analysis of the San Jose 311 call:

1.  **PROBLEM:**
    The core problem is a resident's discomfort due to the presence of two individuals near a park entrance who they believe might be experiencing homelessness. While the individuals are not causing any immediate danger, blocking access, or creating trash, their presence is making the area "feel uncomfortable" to the caller. This represents a perceived issue of public order and community comfort rather than a direct civic violation or safety threat.

2.  **IMPACT:**
    Based solely on the call/message, the primary impact is a **community comfort/perception impact**. The resident explicitly states, "The area just feels uncomfortable," indicating a negative psychological or experiential effect on a community member using or passing through the park. There is no stated impact on public health, safety (explicitly denied by the caller), or service disruption.

3.  **URGENCY:**
    **LOW**
    The resident explicitly states, "They are not blocking anyone, and I do not see trash or danger." There is no indication of immediate threat to life, property, or public safety, nor is there a critical service disruption. The issue is one of perceived discomfort rather than an emergency.

4.  **ACTION:**
    *   **Primary human responder:** A **Homeless Outreach Team** (or a non-emergency mobile crisis/community response team focused on unhoused individuals) should review this first. This team is equipped to engage with individuals who may be experiencing homelessness in a non-enforcement capacity, assess their needs, and offer resources.
    *   **Supporting services:** Based *only* on the report, no specific supporting services are clearly indicated. The report does not mention mental health crises, substance abuse, specific health needs, or significant park rule violations (beyond "sitting near the entrance"). Therefore, the outreach team would primarily focus on the immediate individuals and their situation. If the outreach team identifies a specific park rule violation that needs attention *after* their initial assessment (e.g., an established encampment, though not stated here), they might consult **Parks and Recreation staff**.
    *   **Precautions:** Responders should approach the situation with empathy and a non-confrontational attitude, recognizing that the individuals are not reported to be causing harm or disruption. They should prioritize assessing the situation upon arrival for any actual safety concerns (which are not reported by the caller) and offer support and resources to the individuals if they are indeed unhoused and receptive, while respecting their rights and privacy.

=== RESIDENT RESPONSE ===
Thank you for your report regarding people sitting near the park entrance causing discomfort. We've received it and a human dispatcher will review the information. Appropriate city services may be considered based on the details provided. For non-emergency issues, please expect a review within 3-5 business days.

The prompt/output was acceptable as this situation is pretty vague making it hard to create a clear action and category alongside it not being as much of an emergency yet, AI was able to recognize this low urgency situation and the proper actions for it.